In [1]:
import torch 
import torch.nn as nn


In [2]:
samples = 50
split = int(samples * 0.9)
features = 5

x_0 = torch.randn(samples, features) + torch.tensor([1.4, 3.7, 2.2, -2.34, -1.3])
y_0 = torch.zeros(samples, dtype=torch.long)

x_1 = torch.randn(samples, features) + torch.tensor([-0.3, -2.34, 1.23, -0.9, 2.34])
y_1 = torch.ones(samples, dtype=torch.long)

train_x = torch.cat([x_0[:split, :], x_1[:split, :]], dim=0)
train_y = torch.cat([y_0[:split], y_1[:split]], dim=0)

test_x = torch.cat([x_0[split:,:], x_1[split:, :]], dim=0)
test_y = torch.cat([y_0[split:], y_1[split:]], dim=0)

print(train_x.shape)
print(train_y.shape)
print(test_x.shape)
print(test_y.shape)


torch.Size([90, 5])
torch.Size([90])
torch.Size([10, 5])
torch.Size([10])


In [3]:
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

class DummyDataset(Dataset):
    def __init__(self, x, y):
        self.features = x
        self.labels = y

    def __getitem__(self, index):
        return self.features[index], self.labels[index]

    def __len__(self):
        return self.features.shape[0]
    
train_ds = DummyDataset(train_x, train_y)
test_ds = DummyDataset(test_x, test_y)

train_loader = DataLoader(
        dataset = train_ds,
        batch_size = 4,
        shuffle=True,
        drop_last=True
)

test_loader = DataLoader(
        dataset=test_ds,
        batch_size=4,
        shuffle=False
)

In [4]:
class NeuralNetwork(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()

        self.mlp = nn.Sequential(
            nn.Linear(d_in, 50),
            nn.ReLU(),
            nn.Linear(50, d_out)
        )

    def forward(self, x):
        logits = self.mlp(x)
        return logits

In [7]:
import torch.nn.functional as F

model = NeuralNetwork(features, 2)
model = model.to("cuda")
optimizer= torch.optim.SGD(model.parameters(), lr=0.01)

num_epochs = 100

for epoch in range(num_epochs):
    model.train()
    for batch_idx, (x, y) in enumerate(train_loader):
        x, y = x.to("cuda"), y.to("cuda")
        logits = model(x)
        loss = F.cross_entropy(logits, y)
    
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
        print(f"Epoch: {epoch+1}/{num_epochs}| Batch:{batch_idx+1}/{len(train_loader)}| Loss: {loss:.2f}")

Epoch: 1/100| Batch:1/22| Loss: 0.58
Epoch: 1/100| Batch:2/22| Loss: 0.58
Epoch: 1/100| Batch:3/22| Loss: 0.49
Epoch: 1/100| Batch:4/22| Loss: 0.50
Epoch: 1/100| Batch:5/22| Loss: 0.50
Epoch: 1/100| Batch:6/22| Loss: 0.30
Epoch: 1/100| Batch:7/22| Loss: 0.38
Epoch: 1/100| Batch:8/22| Loss: 0.39
Epoch: 1/100| Batch:9/22| Loss: 0.22
Epoch: 1/100| Batch:10/22| Loss: 0.20
Epoch: 1/100| Batch:11/22| Loss: 0.23
Epoch: 1/100| Batch:12/22| Loss: 0.27
Epoch: 1/100| Batch:13/22| Loss: 0.16
Epoch: 1/100| Batch:14/22| Loss: 0.33
Epoch: 1/100| Batch:15/22| Loss: 0.30
Epoch: 1/100| Batch:16/22| Loss: 0.25
Epoch: 1/100| Batch:17/22| Loss: 0.15
Epoch: 1/100| Batch:18/22| Loss: 0.21
Epoch: 1/100| Batch:19/22| Loss: 0.27
Epoch: 1/100| Batch:20/22| Loss: 0.30
Epoch: 1/100| Batch:21/22| Loss: 0.16
Epoch: 1/100| Batch:22/22| Loss: 0.26
Epoch: 2/100| Batch:1/22| Loss: 0.08
Epoch: 2/100| Batch:2/22| Loss: 0.16
Epoch: 2/100| Batch:3/22| Loss: 0.17
Epoch: 2/100| Batch:4/22| Loss: 0.13
Epoch: 2/100| Batch:5/22|

In [8]:
def compute_accuracy(model, dataloader):
    model = model.eval()
    correct = 0.
    num_samples = 0

    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to("cuda"), y.to("cuda")
            logits = model(x)
            pred = torch.argmax(logits, dim=1)
            compare = pred == y
            correct+=torch.sum(compare)
            num_samples+=len(compare)

    return (correct / num_samples).item()

compute_accuracy(model, test_loader)

1.0